# Chapter 4 — GPT 모델 조립하기 : 코드 해설 노트북 (Python 초보용)

> `Chapter_4_Excercise_GPT.ipynb` 의 원본 코드 셀은 그대로 두고, 번호가 붙은 셀마다 바로 위에 설명을 넣은 해설판입니다.

## 이 챕터에서 하는 일 (한 문장)

Chapter 2 의 **데이터 준비**, Chapter 3 의 **어텐션**에 몇 가지 부품을 더해 **GPT 모델 전체를 조립**하고, 글을 한 토큰씩 이어 쓰게 해 봅니다.

## 전체 구조 지도

```
 token IDs  [batch, T]
     |
     v
 token embedding + position embedding        (Chapter 2)
     |
     v
 +-------------------------------+
 |  TransformerBlock   x 12      |
 |   LayerNorm                   |
 |   MultiHeadAttention          |   (Chapter 3)
 |   + shortcut                  |
 |   LayerNorm                   |
 |   FeedForward (GELU)          |
 |   + shortcut                  |
 +-------------------------------+
     |
     v
 final LayerNorm
     |
     v
 out_head (Linear)  ->  logits  [batch, T, vocab_size]
```

**로짓(logits)** 은 "다음 토큰으로 단어장의 각 토큰이 올 것 같은 정도"를 나타내는 **점수**입니다. 5만여 개 토큰 각각에 점수가 하나씩 매겨집니다.

## 셀 구성

| 셀 | 내용 | 비고 |
|---|---|---|
| 1 | `GPTDatasetV1`, `create_dataloader_v1` | Chapter 2 복습 |
| 2 | `MultiHeadAttention` | Chapter 3 복습 |
| 3 | `LayerNorm`, `GELU`, `FeedForward` | 새 부품 3개 |
| 4 | `TransformerBlock`, `GPTModel` | ⭐ 부품 조립 |
| 5 | `generate_text_simple` | 글 생성 함수 |
| 6 | `main` | 실제로 실행해 보기 |

---
## 셀 1 — (복습) 데이터셋과 데이터로더

### 🎯 이 셀이 하는 일
Chapter 2 에서 만든 `GPTDatasetV1` 클래스와 `create_dataloader_v1` 함수를 그대로 다시 적어 둔 것입니다. 이 노트북 하나만으로 실행할 수 있게 모아 둔 것이며, **새로운 내용은 없습니다.**

### 30초 복습

```python
import tiktoken
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
```
- `import torch.nn as nn` : `torch.nn` 을 앞으로 `nn` 이라는 **짧은 별명**으로 부르겠다는 뜻입니다. `nn.Linear`, `nn.Module` 처럼 씁니다.
- `from A import B` : A 모듈에서 B 만 콕 집어 가져옵니다.

| 이름 | 역할 |
|---|---|
| `GPTDatasetV1.__init__` | 글을 토큰화한 뒤 슬라이딩 윈도우로 (입력, 정답) 쌍을 만들어 보관 |
| `input_chunk = token_ids[i : i+max_length]` | 입력 조각 |
| `target_chunk = token_ids[i+1 : i+max_length+1]` | 정답 조각 = 입력을 **한 칸 민 것** |
| `__len__`, `__getitem__` | 샘플 개수 / idx 번째 쌍 돌려주기 |
| `create_dataloader_v1` | 토크나이저 → Dataset → DataLoader 를 한 번에 만들어 줌 |

Chapter 2 버전과 달리 길이를 검사하는 `assert` 줄만 빠져 있고 나머지는 같습니다. 자세한 설명은 `Chapter_2_Exercise_Dataset.explain.ipynb` 의 셀 4, 5 를 참고하세요.

> 이 노트북의 `main()`(셀 6)은 문장 하나만 직접 넣어 보기 때문에 이 셀의 코드를 실제로 쓰지는 않습니다. Chapter 5(학습)에서 쓰입니다.

In [7]:
# 1
import tiktoken
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

#####################################
# Chapter 2: 데이터 로딩 및 처리
#####################################

class GPTDatasetV1(Dataset):
    """
    GPT 학습을 위한 데이터셋 클래스입니다.
    텍스트를 입력받아 토큰화하고, 입력(input)과 타겟(target) 쌍을 만듭니다.
    """
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # 1. 전체 텍스트를 토큰화합니다.
        # <|endoftext|> 같은 특수 토큰도 허용하여 인코딩합니다.
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # 2. 슬라이딩 윈도우 방식으로 데이터를 조각냅니다.
        # stride만큼 이동하면서 max_length 길이의 덩어리(chunk)를 만듭니다.
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            # 타겟은 입력보다 한 칸 뒤의 토큰들입니다 (다음 토큰 예측 과제).
            target_chunk = token_ids[i + 1: i + max_length + 1]
            
            # 텐서로 변환하여 저장
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        # 데이터셋의 총 샘플 수 반환
        return len(self.input_ids)

    def __getitem__(self, idx):
        # 특정 인덱스의 입력과 타겟 쌍 반환
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True, num_workers=0):
    """
    텍스트 데이터를 받아 학습에 사용할 DataLoader를 생성하는 함수입니다.
    """
    # 토크나이저 초기화 (GPT-2용 BPE 인코딩 사용)
    tokenizer = tiktoken.get_encoding("gpt2")

    # 데이터셋 생성
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # 데이터로더 생성 (배치 단위로 데이터를 묶어주고 셔플링 수행)
    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

    return dataloader


---
## 셀 2 — (복습) `MultiHeadAttention`

### 🎯 이 셀이 하는 일
Chapter 3 셀 4에서 만든 효율적인 멀티 헤드 어텐션을 한글 주석과 함께 다시 적어 둔 것입니다. 코드 속 주석의 1~9번 순서가 곧 계산 순서입니다.

### 30초 복습 — `forward` 9단계

| 단계 | 코드 | 한 줄 요약 |
|---|---|---|
| 1 | `self.W_key(x)` 등 | Q, K, V 만들기 |
| 2 | `.view(b, num_tokens, num_heads, head_dim)` | 헤드 수만큼 쪼개기 (모양만 변경) |
| 3 | `.transpose(1, 2)` | 헤드 축을 앞으로 |
| 4 | `queries @ keys.transpose(2, 3)` | 토큰끼리 관련도 점수 |
| 5 | `masked_fill_(mask_bool, -torch.inf)` | 미래 가리기 |
| 6 | `torch.softmax(... / head_dim**0.5)` + dropout | 점수 → 비율(합 1) |
| 7 | `attn_weights @ values` | 비율대로 정보 섞기 |
| 8 | `.contiguous().view(b, num_tokens, d_out)` | 헤드들 다시 합치기 |
| 9 | `self.out_proj(...)` | 마무리 선형 레이어 |

**입력과 출력의 모양이 같다**는 점이 중요합니다 : `[b, T, emb_dim]` → `[b, T, emb_dim]`. 덕분에 이 부품을 여러 번 겹쳐 쌓을 수 있습니다.

자세한 설명은 `Chapter_3_Excercise_Attention.explain.ipynb` 의 셀 1, 4 를 참고하세요.

In [8]:
# 2

#####################################
# Chapter 3: 어텐션 메커니즘 (모델의 핵심)
#####################################
class MultiHeadAttention(nn.Module):
    """
    멀티 헤드 셀프 어텐션 (Multi-Head Self-Attention) 모듈입니다.
    입력 데이터 간의 관계성을 여러 관점(Head)에서 병렬로 학습합니다.
    """
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "출력 차원(d_out)은 헤드 수(num_heads)로 나누어 떨어져야 합니다."

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads  # 각 헤드가 담당할 차원 크기

        # Query, Key, Value를 만들기 위한 선형 투영 레이어들
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        
        # 멀티 헤드 결과를 하나로 합친 후 통과시키는 출력 레이어
        self.out_proj = nn.Linear(d_out, d_out) 
        self.dropout = nn.Dropout(dropout)
        
        # Causal Mask (인과적 마스킹) 생성: 미래의 토큰을 보지 못하게 함
        # 상삼각 행렬(대각선 위쪽)을 1로 채워서 나중에 마스킹에 사용
        self.register_buffer("mask",
                             torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape # b: 배치 크기, num_tokens: 시퀀스 길이

        # 1. Q, K, V 계산
        keys = self.W_key(x)     # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # 2. 헤드 나누기 (Multi-head splitting)
        # 차원을 변형하여 여러 헤드가 병렬로 처리할 수 있게 함
        # (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # 3. 차원 순서 변경 (Transpose)
        # (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        # 이렇게 하면 (num_tokens, head_dim) 행렬이 헤드 개수만큼 독립적으로 존재하게 됨
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # 4. Scaled Dot-Product Attention 계산
        # Query와 Key의 내적 (유사도 계산)
        attn_scores = queries @ keys.transpose(2, 3)  # 결과 Shape: (b, num_heads, num_tokens, num_tokens)

        # 5. 마스킹 (Masking)
        # 현재 시점보다 미래의 토큰 정보를 참조하지 못하게 -inf로 가림
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        # 6. 소프트맥스 및 드롭아웃
        # 점수를 확률로 변환 (합이 1이 되도록)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # 7. Value와의 가중치 합 (Context Vector 계산)
        # Shape: (b, num_heads, num_tokens, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 

        # 8. 헤드 결합 (Concatenation)
        # 나눠졌던 헤드들을 다시 원래의 d_out 차원으로 합침
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        
        # 9. 최종 선형 투영
        context_vec = self.out_proj(context_vec) 

        return context_vec


---
## 셀 3 — 새 부품 3개 : `LayerNorm`, `GELU`, `FeedForward`

### 🎯 이 셀이 하는 일
트랜스포머 블록을 만드는 데 필요한 작은 부품 세 개를 정의합니다.

| 부품 | 한 줄 역할 | 비유 |
|---|---|---|
| `LayerNorm` | 숫자들의 크기를 고르게 맞춤 | 과목마다 다른 점수를 "표준 점수"로 환산 |
| `GELU` | 단순 계산에 "굴곡"을 넣음 | 음수는 거의 끄고 양수는 통과시키는 부드러운 스위치 |
| `FeedForward` | 각 토큰의 벡터를 한 번 더 가공 | 어텐션이 모아 온 정보를 혼자 곱씹는 시간 |

---
### ① `LayerNorm` — 층 정규화

**왜 필요할까?** 층을 여러 개 쌓으면 숫자가 층을 지날 때마다 점점 커지거나 작아져 학습이 불안정해집니다. 그래서 중간중간 **평균 0, 분산 1** 로 다시 맞춰 줍니다.

```python
def __init__(self, emb_dim):
    super().__init__()
    self.eps = 1e-5
    self.scale = nn.Parameter(torch.ones(emb_dim))
    self.shift = nn.Parameter(torch.zeros(emb_dim))
```
- `1e-5` : 0.00001 (10의 −5제곱)을 쓰는 방법입니다.
- `nn.Parameter(...)` : "이 텐서는 **학습으로 바뀌는 값**"이라고 등록합니다. `scale` 은 1로, `shift` 는 0으로 시작합니다.

```python
def forward(self, x):
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    norm_x = (x - mean) / torch.sqrt(var + self.eps)
    return self.scale * norm_x + self.shift
```

| 코드 | 설명 |
|---|---|
| `x.mean(dim=-1, ...)` | 마지막 축(= 토큰 하나의 벡터 안)에서 **평균** |
| `x.var(dim=-1, ...)` | 같은 방향으로 **분산**(값들이 퍼진 정도) |
| `keepdim=True` | 계산 후에도 축을 없애지 않고 크기 1로 남김 → 원래 `x` 와 바로 빼기/나누기 가능 |
| `unbiased=False` | 분산을 n−1 이 아니라 n 으로 나눠 구함 (원래 GPT-2 방식과 맞추기 위함) |
| `(x - mean) / torch.sqrt(var + eps)` | 평균을 빼고 표준편차로 나눔 → 평균 0, 분산 1 |
| `+ self.eps` | 분산이 0 일 때 **0 으로 나누는 사고**를 막는 아주 작은 수 |
| `self.scale * norm_x + self.shift` | 모델이 필요하면 크기와 위치를 다시 조절할 수 있게 해 주는 학습 파라미터 |

작은 예 :
```
 x      = [ 2,  4,  6]      mean = 4,  std ~ 1.63
 norm_x = [-1.22, 0, 1.22]  mean = 0,  var = 1
```

---
### ② `GELU` — 활성화 함수

**왜 필요할까?** `nn.Linear` 같은 선형 계산만 계속 쌓으면, 아무리 많이 쌓아도 결국 선형 계산 **한 번**과 같아집니다. 중간에 **굽은(비선형) 함수**를 넣어야 복잡한 패턴을 배울 수 있습니다. 이런 함수를 **활성화 함수**라고 합니다.

```python
def forward(self, x):
    return 0.5 * x * (1 + torch.tanh(
        torch.sqrt(torch.tensor(2.0 / torch.pi)) *
        (x + 0.044715 * torch.pow(x, 3))
    ))
```
수식은 복잡해 보이지만 **외울 필요가 없습니다.** GELU 를 빠르게 계산하기 위한 근사 공식을 그대로 옮긴 것입니다. 동작만 기억하세요.

```
 input  :  -3     -1     0     1     3
 output :  ~0   -0.16    0   0.84   ~3
```
- 큰 양수 → 거의 그대로 통과
- 큰 음수 → 거의 0
- 0 근처 → 부드럽게 이어짐

`torch.tanh`, `torch.sqrt`, `torch.pow(x, 3)`(세제곱), `torch.pi`(원주율)는 수학 함수/상수입니다. 학습되는 값이 없어서 `__init__` 에는 `super().__init__()` 만 있습니다.

---
### ③ `FeedForward` — 피드 포워드 네트워크

```python
def __init__(self, cfg):
    super().__init__()
    self.layers = nn.Sequential(
        nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
        GELU(),
        nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
    )

def forward(self, x):
    return self.layers(x)
```
- `cfg` : 설정값이 담긴 **딕셔너리**. `cfg["emb_dim"]` 은 그 안에서 `"emb_dim"` 이라는 이름표의 값을 꺼냅니다(GPT-2 Small 은 768).
- `nn.Sequential(a, b, c)` : 레이어들을 **순서대로 줄 세워** 하나로 묶습니다. 입력이 a → b → c 를 차례로 통과합니다.

```
 768  --Linear-->  3072  --GELU-->  3072  --Linear-->  768
       (x4 넓히기)                        (다시 줄이기)
```
넓은 공간으로 펼쳐서 가공한 뒤 원래 크기로 되돌립니다. **입출력 크기가 같으므로** 역시 여러 번 쌓을 수 있습니다. 어텐션이 "토큰끼리 정보를 주고받는" 단계라면, 피드 포워드는 "각 토큰이 받은 정보를 **혼자** 소화하는" 단계입니다.

> 이 셀은 정의만 하므로 출력이 없습니다.

In [9]:
# 3

#####################################
# Chapter 4: GPT 아키텍처 구성 요소
#####################################
class LayerNorm(nn.Module):
    """
    층 정규화 (Layer Normalization): 학습 안정성을 높임
    """
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim)) # 학습 가능한 스케일 파라미터 (Gamma)
        self.shift = nn.Parameter(torch.zeros(emb_dim)) # 학습 가능한 시프트 파라미터 (Beta)

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        # TODO: LayerNorm 계산식의 통계량 변수를 채우세요.
        # 힌트: 평균(mean)과 분산(var)을 사용해 정규화합니다.
        # norm_x = (x - ????) / torch.sqrt(???? + self.eps)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        # TODO: LayerNorm 학습 파라미터를 채우세요.
        # 힌트: scale로 곱하고 shift를 더하는 형태를 사용합니다.
        # return self.???? * norm_x + self.????
        return self.scale * norm_x + self.shift


class GELU(nn.Module):
    """
    GELU 활성화 함수: GPT 계열에서 주로 사용하는 비선형 함수
    """
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    """
    피드 포워드 네트워크 (Feed-Forward Network)
    어텐션이 모은 정보를 각 토큰별로 개별적으로 가공하는 역할
    보통 임베딩 차원을 4배로 늘렸다가 다시 줄임
    """
    def __init__(self, cfg):
        super().__init__()
        # TODO: FeedForward의 확장 배수와 활성화 함수 클래스를 채우세요.
        # 힌트: GPT 구조에서는 hidden 크기를 4배로 확장합니다. 확장 Linear와 축소 Linear 사이에 GELU 활성화 함수를 넣으세요.
        self.layers = nn.Sequential(
            # nn.Linear(cfg["emb_dim"], ???? * cfg["emb_dim"]),
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            # ????(),
            GELU(),
            # nn.Linear(???? * cfg["emb_dim"], cfg["emb_dim"]),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)



---
## 셀 4 — `TransformerBlock` 과 `GPTModel` (⭐ 가장 중요)

### 🎯 이 셀이 하는 일
지금까지 만든 부품을 조립합니다.
- `TransformerBlock` : 어텐션 + 피드 포워드를 묶은 **한 층**
- `GPTModel` : 임베딩 + 블록 12개 + 출력 헤드 = **완성된 GPT**

---
### ① `TransformerBlock`

**`__init__` — 부품 꺼내 놓기**
```python
self.att = MultiHeadAttention(
    d_in=cfg["emb_dim"], d_out=cfg["emb_dim"],
    context_length=cfg["context_length"],
    num_heads=cfg["n_heads"],
    dropout=cfg["drop_rate"], qkv_bias=cfg["qkv_bias"])
self.ff = FeedForward(cfg)
self.norm1 = LayerNorm(cfg["emb_dim"])
self.norm2 = LayerNorm(cfg["emb_dim"])
self.drop_shortcut = nn.Dropout(cfg["drop_rate"])
```
`d_in=...` 처럼 **이름을 붙여** 인자를 넘기면(키워드 인자) 순서를 틀릴 걱정이 없고 읽기도 쉽습니다. 정규화는 두 군데에 쓰므로 `norm1`, `norm2` 두 개를 따로 만듭니다(각자 학습되는 값이 다르기 때문).

**`forward` — 두 번의 같은 패턴**
```python
# 전반부 : 어텐션
shortcut = x              # (1) 원본을 따로 보관
x = self.norm1(x)         # (2) 정규화
x = self.att(x)           # (3) 어텐션
x = self.drop_shortcut(x) # (4) 드롭아웃
x = x + shortcut          # (5) 보관해 둔 원본을 더함

# 후반부 : 피드 포워드 (att 자리에 ff 만 바뀜)
shortcut = x
x = self.norm2(x)
x = self.ff(x)
x = self.drop_shortcut(x)
x = x + shortcut
```

```
 x ---+---> norm1 -> att -> dropout ---(+)---+---> norm2 -> ff -> dropout ---(+)---> out
      |                                 ^    |                                ^
      +----------- shortcut ------------+    +----------- shortcut -----------+
```

**숏컷(shortcut, 잔차 연결)이 왜 필요할까?**
- `x = x + shortcut` 은 "가공한 결과에 **원본을 그대로 더해** 준다"는 뜻입니다.
- 층을 깊게 쌓으면 학습 신호(기울기)가 뒤에서 앞으로 전달되다가 점점 약해져 앞쪽 층이 배우지 못하는 문제(**기울기 소실**)가 생깁니다. 숏컷은 신호가 건너뛸 수 있는 **지름길**이 되어 이를 막아 줍니다.
- 각 층은 "처음부터 새로 만들기"가 아니라 "**원본에 조금 보태기**"만 하면 되므로 배우기도 쉬워집니다.

정규화를 어텐션/피드 포워드 **앞**에 두는 이 방식을 Pre-LayerNorm 이라고 합니다.

---
### ② `GPTModel`

**`__init__` — 부품 꺼내 놓기**
```python
self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])        # 50257 x 768 표
self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])    # 1024 x 768 표
self.drop_emb = nn.Dropout(cfg["drop_rate"])

self.trf_blocks = nn.Sequential(
    *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

self.final_norm = LayerNorm(cfg["emb_dim"])
self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
```
- `tok_emb` : 토큰 ID → 벡터 (행 수 = **단어장 크기**)
- `pos_emb` : 자리 번호 → 벡터 (행 수 = **최대 문맥 길이**)
- `*[ ... for _ in range(n_layers)]` : 리스트 컴프리헨션으로 블록 12개를 만들고, 맨 앞의 **`*` 가 리스트를 풀어서** 낱개 인자로 넘깁니다. 즉 `nn.Sequential(블록1, 블록2, ..., 블록12)` 와 같습니다.
- `out_head` : 길이 768 벡터를 **단어장 크기(50257)만큼의 점수**로 바꾸는 마지막 레이어

**`forward` — 계산 순서**
```python
def forward(self, in_idx):
    batch_size, seq_len = in_idx.shape
    tok_embeds = self.tok_emb(in_idx)
    pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
    x = tok_embeds + pos_embeds
    x = self.drop_emb(x)
    x = self.trf_blocks(x)
    x = self.final_norm(x)
    logits = self.out_head(x)
    return logits
```

| 줄 | 하는 일 | 모양 |
|---|---|---|
| `in_idx` | 입력 토큰 ID | `[b, T]` |
| `self.tok_emb(in_idx)` | 토큰 임베딩 | `[b, T, 768]` |
| `self.pos_emb(torch.arange(seq_len, ...))` | 0 ~ T−1 번 자리의 위치 임베딩 | `[T, 768]` |
| `tok_embeds + pos_embeds` | 더하기 (브로드캐스팅) | `[b, T, 768]` |
| `self.trf_blocks(x)` | 블록 12개 통과 | `[b, T, 768]` |
| `self.final_norm(x)` | 마지막 정규화 | `[b, T, 768]` |
| `self.out_head(x)` | 단어장 전체에 대한 점수 | `[b, T, 50257]` |

`device=in_idx.device` : 위치 번호 텐서를 **입력과 같은 장치**(CPU 또는 GPU)에 만들라는 뜻입니다. 서로 다른 장치에 있는 텐서끼리는 계산할 수 없기 때문입니다.

### ⭐ 출력 `[b, T, 50257]` 의 의미
입력의 **모든 자리마다** "이 다음에 올 토큰" 점수가 5만여 개씩 나옵니다.
```
 input  :  Hello   ,     I     am
 output :  [..]   [..]  [..]  [..]      each [..] = 50257 scores
            |      |     |     |
          next?  next? next? next?  <- the LAST one predicts the new token
```
학습할 때는 모든 자리의 예측을 정답과 비교하고(Chapter 5), 글을 생성할 때는 **마지막 자리**의 점수만 씁니다(셀 5).

> 이 셀도 정의만 하므로 출력이 없습니다.

In [10]:
# 4
##중요 : 아래는 GPT 모델 전체 구조를 정의하는 코드입니다

#####################################
# Chapter 4: GPT 아키텍처 조립
#####################################
class TransformerBlock(nn.Module):
    """
    표준 트랜스포머 블록 (Decoder Block)
    구조: LayerNorm -> Attention -> Add(Residual) -> LayerNorm -> FeedForward -> Add(Residual)
    """
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # region [어텐션 블록 (Residual Connection 적용)]
        shortcut = x
        x = self.norm1(x) # Pre-LayerNorm 방식
        # TODO
        # x = self.????(x)   
        x = self.att(x)   
        x = self.drop_shortcut(x)
        x = x + shortcut  # 원본 입력을 더해줌 (기울기 소실 방지)
        # endregion

        # region [피드 포워드 블록 (Residual Connection 적용)]
        shortcut = x
        x = self.norm2(x)
        # TODO
        # x = self.????(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # 원본 입력 다시 더함
        # endregion

        return x
class GPTModel(nn.Module):
    """
    전체 GPT 모델 구조 정의
    Embedding -> Transformer Blocks -> Final Norm -> Output Head
    """
    def __init__(self, cfg):
        super().__init__()
        # 토큰 임베딩 (단어 -> 벡터)
        # TODO
        # self.tok_emb = nn.Embedding(cfg["????"], cfg["emb_dim"])
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        # 위치 임베딩 (위치 정보 -> 벡터)
        # TODO 
        # self.pos_emb = nn.Embedding(cfg["????"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        # 트랜스포머 블록 쌓기 (n_layers 만큼)
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        # 최종 정규화 및 출력 헤드
        self.final_norm = LayerNorm(cfg["emb_dim"])
        # TODO
        # self.out_head = nn.Linear(cfg["emb_dim"], cfg["????"], bias=False)
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        
        # region [임베딩 생성]
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        # endregion

        # region [토큰 임베딩과 위치 임베딩 합산]
        # TODOe
        # x = tok_embeds + ????
        x = tok_embeds + pos_embeds
        # endregion

        x = self.drop_emb(x)
        
        # region [트랜스포머 블록 통과]
        # TODO
        # x = self.????(x)
        x = self.trf_blocks(x)
        # endregion

        # region [최종 출력 계산]
        x = self.final_norm(x)
        # TODO
        # logits = self.????(x) # 각 단어에 대한 예측 점수 (Logits)
        logits = self.out_head(x) # 각 단어에 대한 예측 점수 (Logits)
        # endregion
        
        return logits

---
## 셀 5 — `generate_text_simple` : 글 이어 쓰기

### 🎯 이 셀이 하는 일
모델에게 시작 문장을 주고 **다음 토큰을 하나 예측 → 문장 뒤에 붙임 → 다시 예측** 을 반복해서 글을 이어 쓰게 합니다.

```
 step 1 :  [Hello , I am]               -> model -> "X1"
 step 2 :  [Hello , I am X1]            -> model -> "X2"
 step 3 :  [Hello , I am X1 X2]         -> model -> "X3"
 ...
```
GPT 가 글을 "한 글자씩 타이핑하듯" 내놓는 이유가 바로 이 반복 구조입니다.

### 코드 뜯어보기

```python
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
```
- `idx` : 지금까지의 토큰 ID 들. 모양 `[배치, 토큰 수]`
- `max_new_tokens` : 새로 만들 토큰 개수 → 그만큼 반복

**① 너무 길면 뒤쪽만 남기기**
```python
idx_cond = idx[:, -context_size:]
```
- 쉼표 앞 `:` → 배치 축은 전부
- 쉼표 뒤 `-context_size:` → 토큰 축은 **뒤에서 context_size 개만**. (음수 인덱스는 "뒤에서부터"입니다)
- 모델은 최대 `context_length` 개까지만 볼 수 있으므로, 글이 길어지면 앞부분은 잘라 냅니다.

**② 모델에게 예측시키기**
```python
with torch.no_grad():
    logits = model(idx_cond)
```
`torch.no_grad()` : 학습하지 않고 결과만 볼 때는 "기울기 계산용 기록"이 필요 없으므로 꺼서 메모리와 시간을 아낍니다.

**③ 마지막 자리의 점수만 꺼내기**
```python
logits = logits[:, -1, :]
```
`[b, T, vocab]` → `[b, vocab]`. 가운데 축에서 `-1`(맨 마지막)만 골랐습니다. 마지막 자리의 예측이 곧 "새로 이어 붙일 토큰"에 대한 예측이기 때문입니다.

**④ 점수가 가장 높은 토큰 고르기**
```python
idx_next = torch.argmax(logits, dim=-1, keepdim=True)
```
- `argmax` : 가장 큰 **값**이 아니라 가장 큰 값이 있는 **위치(번호)** 를 돌려줍니다. 그 번호가 곧 토큰 ID 입니다.
- `keepdim=True` : 결과를 `[b, 1]` 모양으로 유지해 다음 줄에서 바로 이어 붙일 수 있게 합니다.
- 항상 1등만 고르는 이 방식을 **그리디 디코딩(greedy decoding)** 이라고 합니다.

**⑤ 문장 뒤에 붙이기**
```python
idx = torch.cat((idx, idx_next), dim=1)
```
`dim=1`(토큰 축) 방향으로 이어 붙입니다. `[b, T]` + `[b, 1]` → `[b, T+1]`. 이렇게 길어진 `idx` 가 다음 반복의 입력이 됩니다.

반복이 끝나면 `return idx` 로 "원래 토큰 + 새 토큰" 전체를 돌려줍니다.

In [11]:
# 5

def generate_text_simple(model, idx, max_new_tokens, context_size):
    """
    간단한 텍스트 생성 루프
    현재 문맥을 넣어 다음 토큰을 예측하고, 이를 다시 문맥에 추가하여 반복함
    """
    # idx: 현재 문맥의 토큰 인덱스들 (Batch, Time)
    for _ in range(max_new_tokens):

        # 모델이 지원하는 최대 길이(context_size)를 넘지 않도록 자름
        idx_cond = idx[:, -context_size:]

        # 모델 예측 (기울기 계산 불필요)
        # TODO
        # with torch.????():
        with torch.no_grad():
            logits = model(idx_cond)

        # 마지막 타임스텝의 예측값만 가져옴 (다음 단어 예측이므로)
        # (batch, n_token, vocab_size) -> (batch, vocab_size)
        # TODO
        # logits = logits[:, ????, :]
        logits = logits[:, -1, :]

        # 가장 확률(로짓값)이 높은 토큰 선택 (Greedy Decoding)
        # TODO
        # idx_next = torch.????(logits, dim=-1, keepdim=True) 
        idx_next = torch.argmax(logits, dim=-1, keepdim=True) 

        # 예측된 토큰을 현재 시퀀스 뒤에 이어 붙임
        # TODO
        # idx = torch.????((idx, idx_next), dim=1) 
        idx = torch.cat((idx, idx_next), dim=1) 

    return idx


---
## 셀 6 — `main()` : 조립한 모델 실행해 보기

### 🎯 이 셀이 하는 일
GPT-2 Small 크기의 모델을 만들고 "Hello, I am" 뒤에 10개 토큰을 이어 쓰게 합니다. **아직 학습하지 않은 모델**이라는 점이 핵심입니다.

### 코드 뜯어보기

**① 설정 딕셔너리**
```python
GPT_CONFIG_124M = {
    "vocab_size": 50257, "context_length": 1024, "emb_dim": 768,
    "n_heads": 12, "n_layers": 12, "drop_rate": 0.1, "qkv_bias": False
}
```
변수 이름을 **전부 대문자**로 쓴 것은 "바꾸지 않을 설정값(상수)"이라는 관례적 표시입니다. 이 딕셔너리가 셀 3, 4의 `cfg` 로 전달됩니다.

**② 모델 만들기**
```python
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval()
```
- `GPTModel(...)` : 설계도대로 모델을 만듭니다. 안의 숫자(가중치)는 **무작위**로 채워집니다.
- `model.eval()` : 평가 모드 → 드롭아웃이 꺼집니다.

**③ 입력 준비**
```python
tokenizer = tiktoken.get_encoding("gpt2")
encoded = tokenizer.encode(start_context)            # [15496, 11, 314, 716]
encoded_tensor = torch.tensor(encoded).unsqueeze(0)  # shape [1, 4]
```
`unsqueeze(0)` 은 0번 자리에 **크기 1짜리 축을 추가**합니다. `[4]` → `[1, 4]`. 모델은 `[배치, 토큰]` 2차원 입력을 원하므로 "문장 1개짜리 배치"로 만들어 준 것입니다.

**④ 보기 좋게 출력하기**
```python
print(f"\n{50*'='}\n{22*' '}IN\n{50*'='}")
```
- `\n` : 줄바꿈
- `{50*'='}` : `=` 를 50번 반복
- `{22*' '}` : 공백 22칸 (글자 `IN` 을 가운데쯤 놓기 위함)

**⑤ 생성**
```python
out = generate_text_simple(
    model=model, idx=encoded_tensor,
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)
```

**⑥ 숫자를 다시 글로**
```python
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
```
- `squeeze(0)` : `unsqueeze` 의 반대. 크기 1인 0번 축을 없앱니다. `[1, 14]` → `[14]`
- `.tolist()` : 텐서를 파이썬 리스트로 (토크나이저는 리스트를 받습니다)

**⑦ 실행 스위치**
```python
if __name__ == "__main__":
    main()
```
"이 파일을 **직접 실행**했을 때만 `main()` 을 불러라"는 파이썬의 관용 문구입니다. 다른 파일에서 `import` 만 할 때는 실행되지 않습니다. 노트북에서는 항상 참이라 `main()` 이 실행됩니다.

### 실행 결과 읽기
```
Input text: Hello, I am
Encoded input text: [15496, 11, 314, 716]
encoded_tensor.shape: torch.Size([1, 4])
...
Output length: 14
Output text: Hello, I am Featureiman Byeswickattribute argue logger Normandy Compton analogous
```
- `Output length: 14` = 입력 4개 + 새로 만든 10개 ✔
- 이어 쓴 글이 **엉터리**입니다. 실패가 아니라 **정상**입니다. 모델의 구조는 완성됐지만 안의 숫자가 전부 무작위라서 아무 지식이 없기 때문입니다.

```
 Chapter 4 :  structure OK,  weights = random   ->  nonsense text
 Chapter 5 :  train the weights with data       ->  meaningful text
```

In [12]:
# 6
def main():
    # GPT-2 Small 모델 설정값 (124M 파라미터)
    GPT_CONFIG_124M = {
        "vocab_size": 50257,     # 단어 집합 크기
        "context_length": 1024,  # 최대 문맥 길이
        "emb_dim": 768,          # 임베딩 차원
        "n_heads": 12,           # 어텐션 헤드 수
        "n_layers": 12,          # 레이어 수
        "drop_rate": 0.1,        # 드롭아웃 비율
        "qkv_bias": False        # Q,K,V 편향 사용 여부
    }

    torch.manual_seed(123)
    
    # 모델 초기화 및 평가 모드 설정 (드롭아웃 비활성화)
    model = GPTModel(GPT_CONFIG_124M)
    model.eval()

    start_context = "Hello, I am"

    # 입력 텍스트 인코딩
    tokenizer = tiktoken.get_encoding("gpt2")
    encoded = tokenizer.encode(start_context)
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # 배치 차원 추가

    print(f"\n{50*'='}\n{22*' '}IN\n{50*'='}")
    print("\nInput text:", start_context)
    print("Encoded input text:", encoded)
    print("encoded_tensor.shape:", encoded_tensor.shape)

    # 텍스트 생성 실행
    out = generate_text_simple(
        model=model,
        idx=encoded_tensor,
        max_new_tokens=10,  # 10개의 새로운 토큰 생성
        context_size=GPT_CONFIG_124M["context_length"]
    )
    
    # 생성된 결과 디코딩
    decoded_text = tokenizer.decode(out.squeeze(0).tolist())

    print(f"\n\n{50*'='}\n{22*' '}OUT\n{50*'='}")
    print("\nOutput:", out)
    print("Output length:", len(out[0]))
    print("Output text:", decoded_text)

if __name__ == "__main__":
    main()


                      IN

Input text: Hello, I am
Encoded input text: [15496, 11, 314, 716]
encoded_tensor.shape: torch.Size([1, 4])


                      OUT

Output: tensor([[15496,    11,   314,   716, 27018, 24086, 47843, 30961, 42348,  7267,
         49706, 43231, 47062, 34657]])
Output length: 14
Output text: Hello, I am Featureiman Byeswickattribute argue logger Normandy Compton analogous


---
## 📌 Chapter 4 정리

| 부품 | 역할 | 입력 → 출력 모양 |
|---|---|---|
| `tok_emb` + `pos_emb` | 토큰 ID → 벡터 (+ 자리 정보) | `[b, T]` → `[b, T, 768]` |
| `LayerNorm` | 숫자 크기를 고르게 | 변화 없음 |
| `MultiHeadAttention` | 토큰끼리 정보 주고받기 | 변화 없음 |
| `FeedForward` (+`GELU`) | 각 토큰이 정보 소화하기 | 변화 없음 |
| shortcut | 원본을 더해 주는 지름길 | 변화 없음 |
| `out_head` | 벡터 → 단어장 전체 점수(logits) | `[b, T, 768]` → `[b, T, 50257]` |

**꼭 기억할 3가지**
1. GPT = **임베딩 → (정규화·어텐션·피드 포워드) × N층 → 출력 헤드**
2. 블록의 입출력 모양이 같아서 **레고처럼 쌓을 수 있다.**
3. 글 생성 = **"다음 토큰 하나 예측 → 붙이기"의 반복**. 학습 전에는 엉터리 글이 나오는 것이 정상.